# Case Data Analysis

In [1]:
import json
import pandas as pd

In [2]:
DATA_PATH = "../data/data.json"

with open(DATA_PATH) as f:
    data = json.load(f)

In [3]:
df = pd.read_json(DATA_PATH)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10551 entries, 0 to 10550
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   rowid                 10551 non-null  int64         
 1   id                    10551 non-null  str           
 2   citation              10551 non-null  str           
 3   case_name             10551 non-null  str           
 4   case_numbers          10551 non-null  str           
 5   decision_date         10551 non-null  str           
 6   court                 10551 non-null  str           
 7   subject_tags          10551 non-null  str           
 8   source_url            10551 non-null  str           
 9   pdf_url               10551 non-null  str           
 10  court_summary         450 non-null    str           
 11  summary               66 non-null     str           
 12  created_at            10551 non-null  datetime64[us]
 13  has_content           450 n

Convert to Date format

In [5]:
df["decision_date"] = pd.to_datetime(df["decision_date"])

## Subject Tree Analysis

Assumptions:

1. Subtags are separated by or "—" (\u2014) or "–" (\u2013) or "-" or "?"
2. Original subject tags are strings of the form "['tag1 - tag2..', 'tag1 - tag2']"

Issues (solution):

1. Some tags are capitalised (chose to retain as it is)
2. Some tag separators do not contain spaces between the text and separator (retained as it is)
3. Varying whitespace and literal '\n' string (standardised all whitespace and removed '\n' sequence)

In [6]:
import re

def get_taxonomy_tree(tags_list, with_count=False):
    tag_tree = {}
    branches = []
    for tags in tags_list:
        for tag in tags:
            branches.append(tag)

    if not with_count:
        for branch in branches:
            node = tag_tree
            for section in branch:
                node = node.setdefault(section, {})

    else:
        for branch in branches:
            node = tag_tree
            for section in branch:
                if section not in node:
                    node[section] = {"_count": 0, "_children": {}}
                node[section]["_count"] += 1
                node = node[section]["_children"]

    return tag_tree


def convert_to_tag_str(old: str) -> list[list[str]]:
    # We want the form [[tag1, tag2, ...], [tag1, tag2, ...], ...] 
    result = []
    if len(old) < 3:
        return []
    trunc = old[2:-2]
    tags_list = re.split(r", ", trunc)
    tags_list = [tags.strip('"') for tags in tags_list]  # tag branch initially surrounded by quotation marks
    result = [re.split(r" [—–\-\?]+ ", tags) for tags in tags_list]
    result = [[re.sub(r"\n|\t|\\n|\s+", ' ', tag).strip() for tag in tags] for tags in result]
    
    return result
  

In [7]:
df["subject_tags_cleaned"] = df["subject_tags"].apply(convert_to_tag_str)


In [8]:
tags_list = df["subject_tags_cleaned"].to_list()
ori_tags = df["subject_tags"].to_list()

In [9]:
ori_tags[:10]

['["Damages — Assessment — Defamation"]',
 '["Administrative Law — Judicial review — Duty to give reasons — Whether there was a breach of a duty to give reasons"]',
 '["Criminal Procedure and Sentencing — Sentencing — Appeals", "Criminal Law — Statutory offences — Road Traffic Act — Drink driving", "Criminal Procedure and Sentencing — Sentencing — Benchmark sentences", "Criminal Law — Statutory offences — Road Traffic Act — Careless driving for serious offender"]',
 '["Civil Procedure — Appeals", "Abuse of Process — Henderson v Henderson doctrine", "Credit and Security — Money and moneylenders — Illegal moneylending"]',
 '["Trusts — Resulting trusts", "Equity — Estoppel — Proprietary estoppel", "Trusts — Constructive trusts — Common intention constructive trusts", "Trusts — Unlawful trust — Whether trust is unenforceable for illegality"]',
 '["Criminal Procedure and Sentencing — Review", "Criminal Procedure and Sentencing — Stay of execution"]',
 '["Family Law — Ancillary powers of cou

In [10]:
tags_list[:10]

[[['Damages', 'Assessment', 'Defamation']],
 [['Administrative Law',
   'Judicial review',
   'Duty to give reasons',
   'Whether there was a breach of a duty to give reasons']],
 [['Criminal Procedure and Sentencing', 'Sentencing', 'Appeals'],
  ['Criminal Law', 'Statutory offences', 'Road Traffic Act', 'Drink driving'],
  ['Criminal Procedure and Sentencing', 'Sentencing', 'Benchmark sentences'],
  ['Criminal Law',
   'Statutory offences',
   'Road Traffic Act',
   'Careless driving for serious offender']],
 [['Civil Procedure', 'Appeals'],
  ['Abuse of Process', 'Henderson v Henderson doctrine'],
  ['Credit and Security', 'Money and moneylenders', 'Illegal moneylending']],
 [['Trusts', 'Resulting trusts'],
  ['Equity', 'Estoppel', 'Proprietary estoppel'],
  ['Trusts', 'Constructive trusts', 'Common intention constructive trusts'],
  ['Trusts',
   'Unlawful trust',
   'Whether trust is unenforceable for illegality']],
 [['Criminal Procedure and Sentencing', 'Review'],
  ['Criminal Pr

In [11]:
tag_tree = get_taxonomy_tree(tags_list)
tag_tree_with_count = get_taxonomy_tree(tags_list, with_count=True)

In [12]:
with open("../data/subject_tree.json", "w", encoding='utf-8') as f:
    json.dump(tag_tree, f, indent=4)

with open("../data/subject_tree_with_count.json", "w", encoding='utf-8') as f:
    json.dump(tag_tree_with_count, f, indent=4)

In [13]:
def num_subjects_by_depth(tree: dict, depth: int):
    # depth limited search on the subject tree
    if not tree or depth < 1:
        return 1
    sum_leaves = 0
    for branch in tree:
        sum_leaves += num_subjects_by_depth(tree[branch], depth-1)
    
    return sum_leaves

def sort_level(tree: dict) -> list[tuple[str, dict]]:
    concept_list = list(tree.items())
    concept_list.sort(key=lambda x: x[1]["_count"], reverse=True)
    return concept_list

In [14]:
for i in range(1, 5):
    print(f"Found {num_subjects_by_depth(tag_tree, i)} subjects at depth {i}")

Found 673 subjects at depth 1
Found 2692 subjects at depth 2
Found 5284 subjects at depth 3
Found 5853 subjects at depth 4


Top level subject counts

In [15]:
sorted_subjects = sort_level(tag_tree_with_count)
print("Most used subjects in the dataset:", end='\n\n')
for i in range(10):
    print(f"{sorted_subjects[i][1]['_count']} cases | {sorted_subjects[i][0]}")

print('-------------------')
print("Least used subjects in the dataset:", end='\n\n')
for i in range(10):
    print(f"{sorted_subjects[-i-1][1]['_count']} case | {sorted_subjects[-i-1][0]}")

Most used subjects in the dataset:

2458 cases | Civil Procedure
1924 cases | Contract
1536 cases | Criminal Procedure and Sentencing
1266 cases | Criminal Law
1176 cases | Tort
1102 cases | Family Law
746 cases | Companies
648 cases | Arbitration
501 cases | Evidence
483 cases | Insolvency Law
-------------------
Least used subjects in the dataset:

1 case | Malaysia twice per month during weekend access
1 case | Family Law- Section 12 and Section 14 Vulnerable Adults Act 2018
1 case | Damages- Proof of Damages
1 case | Maintenance for Children
1 case | Single Income Marriage
1 case | Spousal Maintenance
1 case | Adverse Inference
1 case | Ancillary Matters
1 case | Physical Abuse
1 case | Sole Custody
